# Chunk-Based Analysis to Reduce Memory Usage, using XArray, HDF5, and Dask

Scientific datasets are increasingly too large to load into memory all at once. High-resolution imaging, large simulation outputs, and long experimental recordings can easily exceed the RAM available on a typical workstation. If analysis requires loading the entire dataset before computation begins, many otherwise simple operations become impossible.

This notebook introduces a different approach: chunk-based analysis. Instead of loading an entire dataset into memory, data can be processed in smaller pieces. Modern scientific Python tools—particularly XArray, NetCDF/HDF5, and Dask—make this possible while keeping the code readable and close to the mathematical operations scientists want to perform. The goal is to understand how these tools cooperate to reduce memory pressure and enable scalable analysis pipelines.

The notebook proceeds in stages. First, you will learn the basic structure of XArray objects, which add labeled dimensions and metadata to NumPy-style arrays. Next, you will see how these arrays can be saved to NetCDF/HDF5 files, enabling efficient on-disk storage with compression and encoding options. Finally, you will explore how lazy loading and chunked computation allow large analyses to run without loading the entire dataset into memory, and how Dask coordinates the computation behind the scenes.

## Setup

### Import Packages

In [ ]:
import dask.distributed
import netCDF4
import numpy as np
import xarray as xr

### Utility Functions

In [ ]:
from contextlib import contextmanager
import os
import sys

import matplotlib.pyplot as plt
from memory_profiler import memory_usage
import pandas as pd


def _generate_calcium_data_file(fname="calcium_imaging_session.nc", nx=256, ny=256, nt=4000, n_cells=12, baseline=0.15, noise_sd=0.05) -> None:
    import numpy as np
    import xarray as xr
    
    time = np.linspace(2, 20, nt)
    
    rng = np.random.default_rng(42)
    
    # coordinate grids
    x = np.arange(nx)
    y = np.arange(ny)
    X, Y = np.meshgrid(x, y, indexing="ij")
    
    # ----------------------------
    # Background signal
    # ----------------------------
    # Start with baseline + pixel noise
    data = baseline + rng.normal(0, noise_sd, size=(nx, ny, nt))
    
    # Add slow global drift over time
    slow_drift = (
        0.03 * np.sin(2 * np.pi * time / 9.0)
        + 0.02 * np.sin(2 * np.pi * time / 3.7 + 1.2)
    )
    data += slow_drift[None, None, :]
    
    # Add a weak spatial background gradient
    spatial_bg = 0.03 * ((X / nx) + (Y / ny))
    data += spatial_bg[:, :, None]
    
    # ----------------------------
    # Shared neuropil-like signal
    # ----------------------------
    shared_events = rng.poisson(0.003, size=nt).astype(float)
    
    kernel_len = 120
    tau = 18
    kernel = np.exp(-np.arange(kernel_len) / tau)
    shared_trace = np.convolve(shared_events, kernel, mode="full")[:nt]
    shared_trace /= shared_trace.max() + 1e-12
    shared_trace *= 0.05
    
    data += shared_trace[None, None, :]
    
    # Add synthetic cells
    cell_masks = []
    cell_traces = []
    
    for i in range(n_cells):
        # random cell center, avoiding borders
        cx = rng.integers(20, nx - 20)
        cy = rng.integers(20, ny - 20)
    
        # random elliptical Gaussian footprint
        sx = rng.uniform(3, 8)
        sy = rng.uniform(3, 8)
        amp = rng.uniform(0.2, 0.8)
    
        footprint = np.exp(-(((X - cx) ** 2) / (2 * sx**2) + ((Y - cy) ** 2) / (2 * sy**2)))
        footprint *= amp
    
        # sparse spike/event train
        event_rate = rng.uniform(0.002, 0.01)
        spikes = rng.poisson(event_rate, size=nt).astype(float)
    
        # calcium decay kernel
        tau_decay = rng.uniform(8, 30)
        kernel_len = 200
        decay_kernel = np.exp(-np.arange(kernel_len) / tau_decay)
    
        trace = np.convolve(spikes, decay_kernel, mode="full")[:nt]
    
        # normalize and scale
        if trace.max() > 0:
            trace = trace / trace.max()
        trace *= rng.uniform(0.3, 1.2)
    
        # add a tiny bit of within-cell temporal noise
        trace += rng.normal(0, 0.01, size=nt)
        trace = np.clip(trace, 0, None)
    
        # add cell contribution to movie
        data += footprint[:, :, None] * trace[None, None, :]
    
        cell_masks.append(footprint)
        cell_traces.append(trace)
    
    
    # Clamp to nonnegative values
    data = np.clip(data, 0, None)
    
    # Build xarray object
    movie = xr.DataArray(
        data,
        name="image",
        dims=["x", "y", "time"],
        coords={
            "x": x,
            "y": y,
            "time": time,
        },
        attrs={
            "description": "Synthetic calcium imaging session with noisy background, drift, shared activity, and localized active cells"
        },
    )
    movie.to_netcdf(fname)



def _format_duration(seconds: float, precision: int = 1) -> str:
    """
    Takes a time in seconds and returns a string (e.g. ) that is more human-readable.

    Looking to do this in a real project?  Some alternatives:
      - `humanize`: https://humanize.readthedocs.io/en/latest/
    """


    if seconds < 0:
        raise ValueError("Duration must be non-negative")

    units = [("s", 1), ("ms", 1e-3), ("µs", 1e-6)]

    for unit, scale in units:
        if seconds >= scale:
            value = seconds / scale
            return f"{value:.{precision}f} {unit}"
    else:
        return f"{seconds / 1e-9:.{precision}f} ns"



@contextmanager
def _trace_lines_of(fun):
    "a (very) basic line tracer.  Collects (timestamp, lineno) for each executed line inside a function."
    import time
    try:
        target_code = fun.__code__
    except AttributeError:
        yield []
        return
    target_frame = None
    records = []

    def tracer(frame, event, arg):
        nonlocal target_frame

        if event == "call" and frame.f_code is target_code:
            target_frame = frame
            return tracer

        elif frame is target_frame:
            if event == "line":
                records.append((frame.f_lineno, time.perf_counter()))
            elif event == "return":
                target_frame = None

            return tracer

    old_trace = sys.gettrace()
    sys.settrace(tracer)

    try:
        yield records
    finally:
        sys.settrace(old_trace)


def _sample_memory(fun, interval=.00005):
    
    # Collect memory traces and line number timings
    with _trace_lines_of(fun) as line_trace:
        memory_trace = memory_usage(fun, interval=interval, timestamps=True)

    # Make Comparable DataFrames out of the two datasets
    line_trace_df = pd.DataFrame(line_trace, columns=['Line', 'Time'])
    if len(line_trace_df) > 0:
        line_trace_df.Time -= line_trace_df.Time[0]
    
    memory_trace = memory_trace[1:]
    
    memory_trace_df = pd.DataFrame(memory_trace, columns=['Memory', 'Time'])
    memory_trace_df['Time'] -= memory_trace_df['Time'][0]
    memory_trace_df['Memory'] -= memory_trace_df['Memory'][0]

    return line_trace_df, memory_trace_df


def _plot_memory(data: pd.DataFrame, x='Time', y='Memory', ax=None):
    "Makes a line plot."

    peak_memory_mb = round(data[y].max(), 1)
    total_time = _format_duration(data[x].max())

    ax = ax if ax is not None else plt.gca()
    ax.plot(data[x], data[y])
    ax.fill_between(data[x], data[y], 0, alpha=0.3)
    ax.set(xlabel='Time (s)', ylabel='Memory (MB)', title=f"Total Time: {total_time} -- Peak Memory: {peak_memory_mb} MB")

    ax.margins(y=0)
    
    ylim_max = data[y].max() * 1.05 if data[y].max() > 1 else 1
    ax.set_ylim(0, ylim_max)
    
    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    return ax


def _plot_line_numbers(data: pd.DataFrame, x='Time', text='Line', linestyle='--', color='gray', alpha=0.3, fontsize=6, ax=None):
    "Makes vertical lines with text above them."
    
    ax = ax if ax is not None else plt.gca()
    ymin, ymax = ax.get_ylim()
    y_text = ymax - 0.04 * (ymax - ymin)

    for _, row in data.iterrows():
        ax.axvline(row[x], linestyle=linestyle, alpha=alpha, color=color)
        ax.text(row[x], y_text, str(int(row[text])), rotation=90, ha="right", va="bottom", fontsize=fontsize)
        
    return ax

def _analyze_memory(*funs, interval=.00005, linestyle='--', color='gray', alpha=0.3, fontsize=6):
    "Convenient wrapper function: records memory traces of provided functions and makes the plot"

    if len(funs) == 1:
        fig, axes = plt.subplots();
        axes = [axes]
    else:
        fig, axes = plt.subplots(nrows=len(funs), sharex=True)


    print(axes)
    for ax, fun in zip(axes, funs):
        lines, memory = _sample_memory(fun, interval=interval)
        # ax = ax if ax is not None else plt.gca()
        _plot_memory(memory, ax=ax)
        _plot_line_numbers(lines, linestyle=linestyle, color=color, alpha=alpha, fontsize=fontsize, ax=ax)
    
    y_max = max([ax.get_ylim()[1] for ax in axes])
    for ax in axes:
        ax.set_ylim(0, y_max)

    plt.tight_layout()


def _format_bytes(bytes: float, precision: int = 2) -> str:
    """
    Takes a time in seconds and returns a string (e.g. ) that is more human-readable.

    Looking to do this in a real project?  Some alternatives:
      - `humanfriendly`: https://pypi.org/project/humanfriendly/#getting-started
    """

    if bytes < 0:
        raise ValueError("bytes must be non-negative")

    units = [("KB", 1000), ("MB", 1_000_000), ("GB", 1_000_000_000), ("TB", 1_000_000_000_000)]

    for unit, scale in reversed(units):
        if bytes >= scale:
            value = bytes / scale
            return f"{value:.{precision}f} {unit}"
    else:
        return f"{bytes} B"

def _file_size(path: str) -> int:
    return os.path.getsize(path)

def _print_file_size(path: str, label='') -> None:
    text = _format_bytes(_file_size(path))
    if label:
        text = label + ': ' + text
    print(text)


class utils:
    analyze_memory = _analyze_memory
    generate_calcium_data_file = _generate_calcium_data_file
    print_file_size = _print_file_size

---

## Section 1: Intro to Working with XArray

NumPy arrays are powerful but minimal: they store numerical data without any built-in information about what the axes represent. In scientific datasets, however, each axis usually corresponds to meaningful quantities such as space, time, wavelength, or experimental condition. When working with multidimensional data, it is often useful to label these dimensions explicitly.

**[XArray](https://docs.xarray.dev/en/stable/user-guide/index.html)** extends NumPy by adding labels and metadata to multidimensional arrays. Each array can have named dimensions, coordinate values, and descriptive metadata attached to it. This makes many operations clearer and safer, since calculations can reference dimensions by name rather than by numeric index. In addition, XArray integrates naturally with file formats such as NetCDF and with distributed computing tools like Dask.



### Exercises

The exercises in this section introduce the core concepts of XArray: treating a DataArray like a NumPy array, labeling dimensions, attaching coordinates, and adding metadata to describe the data.

#### `xr.DataArray()` as a Numpy-Like Array

At its core, an XArray DataArray wraps a NumPy array. Most numerical operations that work on NumPy arrays also work on DataArray objects, including slicing, aggregation, and arithmetic. This means that existing NumPy-style code often requires very little modification to work with labeled arrays.

**Example**: Create a three-dimensional `DataArray` from a Numpy array:

In [ ]:
da = xr.DataArray(
    data=np.random.random(size=(10, 20, 30))
)
da

**Exercise**: Select the first 5 rows of `da`, using the slicing synatax `x[:10, :, :]`

In [ ]:
da[:5, :, :]

**Exercise**: Compute the mean, using either `DataArray.mean()` or `np.mean()`

In [ ]:
da.mean()

In [ ]:
np.mean(da)

**Exercise**: Compute the mean over the third axis, using `da.mean(axis=2)`

In [ ]:
da.mean(axis=2)

#### Labeling the Data and the Dimensions: `name=` and `dims=`

One of XArray’s most useful features is the ability to name dimensions explicitly. Instead of referring to axes by position—such as “axis 0” or “axis 2”—operations can refer to dimensions using meaningful labels like "x", "y", or "time".

Once dimensions are named, many operations become easier to read and harder to misuse. For example, computing the mean across time can be expressed as mean(dim="time"), which clearly communicates the intent of the calculation.

**Exercise**: Make a new `da` 3-dimensional array variable using `xr.DataArray()`, this time additionally setting `name="image"` and `dims=['x', 'y', 'time']`

In [ ]:
da = xr.DataArray(
    data=np.random.random(size=(10, 20, 30)),
    name='image',
    dims=['x', 'y', 'time']
)
da

**Exercise**: Select the fourth time sample using `da.sel(time=4)`

In [ ]:
da.sel(time=4)

**Exercise**: Select the first-through fifth rows by name using `da.sel(x=slice(0, 5))`

In [ ]:
da.sel(x=slice(0, 5))

**Exercise**: Compute the Mean image over time by name, using `da.mean(dim='time')`:

In [ ]:
da.mean(dim='time')

**Exercise**: The time points are stored in the numpy array `t` below.  Use `mask = t > 40; da.sel(time=mask)` to select only the data corresponding to time points greater than 40:

In [ ]:
t = np.linspace(0, 100, 30)

In [ ]:
mask = t > 40
da.sel(time=mask)

#### Labeling each Axis using Coordinates and Attributes

Beyond naming dimensions, XArray allows each axis to have coordinate values that describe the physical meaning of each index. For example, a time axis might correspond to timestamps, or a spatial axis might correspond to pixel positions.

Coordinates make it possible to select data based on meaningful values rather than raw indices. For example, selecting frames after a particular time point can be done using coordinate values instead of computing index positions manually.

**Example**: Run the code below to make a new `da` using `xr.DataArray()`, this time additionally mapping the time axis to the time points themselves using `coords=`:

In [ ]:
da = xr.DataArray(
    data=np.random.random(size=(10, 20, 30)),
    name='image',
    dims=['x', 'y', 'time'],
    coords = {
        'time': np.linspace(0, 100, 30),
    }
)
da

**Exercise**: Use `da.sel(time = slice(40, None))` to select only the data corresponding to time points greater than 40, without first creating a mask:

In [ ]:
da.sel(time = slice(40, None))

#### Adding Descriptions to the data

Support for basic data descriptions is quite extensive.  Things like units, long names for plotting, processing history, and even descriptions for explaining each part of the data is supported by adding the data to a dictionary attached to `xarray` objects called `attrs`.  Some keys are recognized by other tooling (e.g. `units`, `description`, `long_name`), but for the most part, any kind of key-value combination is supported for metadata.

**Example**: Run the code below to create a new `da` DataArray using `DataArray`, this time with extra attributes describing the main variables.

In [ ]:
time = xr.DataArray(
    data = np.linspace(0, 100, 30),
    name = 'time',
    dims=['time'],
    attrs = {
        'units': 's',
        'description': 'time samples for each image frame'
    }
)

da = xr.DataArray(
    data=np.random.random(size=(10, 20, 30)),
    name='image',
    dims=['x', 'y', 'time'],
    coords = {
        'time': time,
    },
    attrs = {
        'units': 'brightness',
        'description': 'A generated random image stack',
        'long_name': 'calcium image pixel brightness',
    }
)
da

**Exercise**: View the attributes of the `da` DataArray with `da.attrs`

In [ ]:
da.attrs

**Exercise**: View the attributes of the `time` coordinate on the `da` DataArray with `da.time.attrs`:

In [ ]:
da.time.attrs

**Exercise**: Plot the mean pixel brightness over time and check that some attributes are used automatically in the plot, with `da.mean(dim=['x', 'y']).plot()`:

In [ ]:
da.mean(dim=['x', 'y']).plot();

There are many, many more features that XArray provides to add convenience to an analysis, but this should be enough to get us started. 

---

## Section 2: Creating HDF5-based NetCDF4 Files with XArray

Once data is organized in an XArray structure, it can easily be saved to disk using scientific file formats such as NetCDF4, which is built on top of the HDF5 storage system. These formats are widely used in scientific computing because they support structured metadata, multidimensional datasets, and efficient storage of large arrays.

Saving data in these formats allows large datasets to be stored and accessed efficiently without requiring them to be fully loaded into memory. It also makes the data portable and accessible to tools outside of Python.

### Exercises

**Exercise**: Use `da.to_netcdf()`, using the `engine='netcdf4'` option, to create an HDF5-compatible file called `example.nc`.

In [ ]:
time = xr.DataArray(
    data = np.linspace(0, 100, 30),
    name = 'time',
    dims=['time'],
    attrs = {
        'units': 's',
        'description': 'time samples for each image frame'
    }
)

da = xr.DataArray(
    data=np.random.random(size=(10, 20, 30)),
    name='image',
    dims=['x', 'y', 'time'],
    coords = {
        'time': time,
    },
    attrs = {
        'units': 'brightness',
        'description': 'A generated random image stack',
        'long_name': 'calcium image pixel brightness',
    }
)
da

In [ ]:
da.to_netcdf('example.nc', engine='netcdf4')

**Exercise**: Open the `example.nc` file in the HDF5 Viewer at https://myhdf5.hdfgroup.org/ to verify it is a valid HDF5 file, and use it to do the following tasks:
  1. View the `time` variable as a line plot.
  2. View the `time` values themselves in a matrix.
  3. View the Image data as a heatmap.
  4. Find the "description" attribute for the `image` variable (hint: check the `inspect` tab) 

#### Changing the `encoding` for a Variable to Save Space using Compression: `zlib` and `complevel`

Large scientific datasets often contain patterns that can be compressed effectively. NetCDF and HDF5 support built-in compression options, which can significantly reduce file size without altering the meaning of the data.

Compression settings such as `zlib` and `complevel` allow the file format to store the data more efficiently on disk. In some cases—particularly for structured or low-entropy data—the resulting file can be much smaller while remaining fully compatible with standard tools.  Other compression libraries are also supported, but here we'll just focus on the big picture: when and where compression helps (all options found [here](https://unidata.github.io/netcdf4-python/#netCDF4.Dataset.createVariable), for the interested)

The exercises in this section explore how compression affects file size for different types of data.

**Example**: Save the `da` DataArray below with two different encodings: one **without** zlib compression, and one **with** zlib compression.  How big of a file size reduction is there?

In [ ]:
da = xr.DataArray(np.random.random(1_000_000), name='data')

In [ ]:
da.to_netcdf('data1.nc', engine='netcdf4')
utils.print_file_size('data1.nc', 'No Compression ')

da.to_netcdf('data2.nc', engine='netcdf4', encoding={'data': {'zlib': True, 'complevel': 4}})
utils.print_file_size('data2.nc', 'Yes Compression', )

**Exercise**:  Save the `da` DataArray below with two different encodings: one **without** zlib compression, and one **with** zlib compression.  How big of a file size reduction is there?

In [ ]:
da = xr.DataArray(np.linspace(0, 10, 1_000_000), name='data')

In [ ]:
da.to_netcdf('data1.nc', engine='netcdf4')
utils.print_file_size('data1.nc', 'No Compression ')

da.to_netcdf('data2.nc', engine='netcdf4', encoding={'data': {'zlib': True, 'complevel': 4}})
utils.print_file_size('data2.nc', 'Yes Compression', )

**Exercise**: Save the `da` DataArray below with two different encodings: one **without** zlib compression, and one **with** zlib compression.  How big of a file size reduction is there?

In [ ]:
da = xr.DataArray(np.arange(1_000_000), name='data')

In [ ]:
da.to_netcdf('data1.nc', engine='netcdf4')
utils.print_file_size('data1.nc', 'No Compression ')

da.to_netcdf('data2.nc', engine='netcdf4', encoding={'data': {'zlib': True, 'complevel': 4}})
utils.print_file_size('data2.nc', 'Yes Compression', )

**Exercise**: Save the `da` DataArray below with two different encodings: one **without** zlib compression, and one **with** zlib compression.  How big of a file size reduction is there?

In [ ]:
da = xr.DataArray(np.zeros(1_000_000), name='data')

In [ ]:
da.to_netcdf('data1.nc', engine='netcdf4')
utils.print_file_size('data1.nc', 'No Compression ')

da.to_netcdf('data2.nc', engine='netcdf4', encoding={'data': {'zlib': True, 'complevel': 4}})
utils.print_file_size('data2.nc', 'Yes Compression', )

---

## Section 3: Analysis Requires Memory: Monitoring Memory Usage for Chained Pipelines

Even when data is stored efficiently on disk, analysis pipelines can still consume large amounts of memory. This is particularly true when multiple operations are chained together, since intermediate results may temporarily allocate large arrays.

In this section, we monitor the memory usage of different analysis pipelines while performing simple computations on an imaging dataset. By observing how memory usage changes over time, it becomes easier to see how different data-loading strategies affect resource consumption.

These experiments illustrate an important principle: the way data is loaded and processed can matter just as much as the analysis itself.

### Exercises

In [ ]:
utils.generate_calcium_data_file('calcium_data.nc')

**Example**: Compute Two Different Mean Projections: one over all pixels, and one over a selection of the frame (a "Region of Interest")

1. Update the functions and run the cell

In [ ]:
def mean_all_data():
    return (
        xr.load_dataarray('calcium_data.nc')
        .mean(dim='time')
    )

def mean_roi_data():
    return (
        xr.load_dataarray('calcium_data.nc')
        .sel(x=slice(0, 100), y=slice(0, 100))
        .mean(dim='time')
    )

2. Check that the functions work: plot each of the generated images.

In [ ]:
mean_all_data().plot()
plt.figure()
mean_roi_data().plot()

3. How much memory do each of the two functions use? Plot a comparison between the two functions.

In [ ]:
utils.analyze_memory(
    mean_all_data,
    mean_roi_data,
)


**Exercise**: Modify the second of the mean-projection functions below to use `xr.open_dataarray()`, which opens the file but doesn't load the data until it is requested.

1. Update the functions and run the cell

In [ ]:
def mean_roi_data():
    return (
        xr.load_dataarray('calcium_data.nc')
        .sel(x=slice(0, 100), y=slice(0, 100))
        .mean(dim='time')
    )

def mean_roi_data2():
    return (
        xr.load_dataarray('calcium_data.nc')
        .sel(x=slice(0, 100), y=slice(0, 100))
        .mean(dim='time')
    )

In [ ]:
def mean_roi_data():
    return (
        xr.load_dataarray('calcium_data.nc')
        .sel(x=slice(0, 100), y=slice(0, 100))
        .mean(dim='time')
    )

def mean_roi_data2():
    return (
        xr.open_dataarray('calcium_data.nc')
        .sel(x=slice(0, 100), y=slice(0, 100))
        .mean(dim='time')
    )

2. Check that the functions still work as before: plot each of the generated images and compare them; despite having different code, they should show the same result.

In [ ]:
mean_roi_data().plot()
plt.figure()
mean_roi_data2().plot()

3. How much memory do each of the two functions use? Plot a comparison between the two functions. Is there a significant diffrence between the two?

In [ ]:
utils.analyze_memory(
    mean_roi_data,
    mean_roi_data2,
)


**Exercise**: Modify the third of the mean-projection functions below to use `xr.open_dataarray(chunks='auto')`, which opens the file, but doesn't load the data until the full computatoin is requested (i.e. add `.compute()`) to the end of the pipeline.

1. Update the functions and run the cell

In [ ]:
def mean_roi_data():
    return (
        xr.load_dataarray('calcium_data.nc')
        .sel(x=slice(0, 100), y=slice(0, 100))
        .mean(dim='time')
    )

def mean_roi_data2():
    return (
        xr.open_dataarray('calcium_data.nc')
        .sel(x=slice(0, 100), y=slice(0, 100))
        .mean(dim='time')
    )


def mean_roi_data3():
    return (
        xr.open_dataarray('calcium_data.nc')
        .sel(x=slice(0, 100), y=slice(0, 100))
        .mean(dim='time')
        .compute()
    )

In [ ]:
def mean_roi_data():
    return (
        xr.load_dataarray('calcium_data.nc')
        .sel(x=slice(0, 100), y=slice(0, 100))
        .mean(dim='time')
    )

def mean_roi_data2():
    return (
        xr.open_dataarray('calcium_data.nc')
        .sel(x=slice(0, 100), y=slice(0, 100))
        .mean(dim='time')
    )


def mean_roi_data3():
    return (
        xr.open_dataarray('calcium_data.nc', chunks='auto')
        .sel(x=slice(0, 100), y=slice(0, 100))
        .mean(dim='time')
        .compute()
    )

2. Check that the the functions all still work as before: plot each of the generated images and compare them; despite having different code, they should show the same result.

In [ ]:
mean_roi_data().plot()
plt.figure()
mean_roi_data2().plot()
plt.figure()
mean_roi_data3().plot()

3. How much memory do each of the three functions use? Plot a comparison between the three functions. Is there a significant diffrence between the three?

In [ ]:
utils.analyze_memory(
    mean_roi_data,
    mean_roi_data2,
    mean_roi_data3,
)


---

## Section 4: Monitoring Dask's Workflow

When Dask executes chunked computations, it constructs a task graph describing how different pieces of the computation depend on each other. A distributed Dask client provides a dashboard that visualizes this process in real time, showing how tasks are scheduled, executed, and completed across workers.

The dashboard allows users to observe how work is distributed, how memory usage changes over time, and how intermediate results flow through the computation graph. This visibility is especially valuable when analyzing large datasets or debugging performance issues.

**Exercise**: Uncomment and run the following code to shift computations out of the process, into a "Distributed Dask" client (note: this will make it so that the `utils.analyze_memory()` function can no longer access the dask-processed data; monitoring will have to be done from the dask workers).  Open the resulting web page and browse through the sections.

In [ ]:
# import dask.distributed
# client = dask.distributed.Client()
# client

**Exercise**: Run the following code over and over, while simultaneously viewing the client monitoring dashboard, looking at different sections of the dashboard, and answer the following questions:
  1. How many workers are processing the data?
  2. Do the workers just break up the work evenly from the beginning, or is there more-complex cooperation happening between them?
  3. How many different tasks did the was the workflow broken down into?
  4. Is memory released between tasks?
  5. Is there a simple relationship between each each task, or is there a more complex compute graph being run?

In [ ]:
(xr.open_dataarray('calcium_data.nc', chunks='auto')
    .sel(x=slice(0, 100), y=slice(0, 100))
    .mean(dim='time')
    .rolling(x=7, center=True).mean()
    .dropna('x')
    .rolling(y=7, center=True).mean()
    .dropna('y')
    .compute()
    .plot()
)